# Bombcell workshop – Open Lab Online, 20 October 2026

**Goal:** go from a Kilosort output folder to a curated set of units, and understand *why* each unit got its label.

1. Run Bombcell
2. Look at the output files
3. Look at units in the GUI
4. Fine-tune the classification
5. Quality metrics vs. stimulus responses: what do we lose by curating?

**Data:** one session from the [chronic Neuropixels 2.0 dataset](https://doi.org/10.5522/04/24411841.v2)
(Lebedeva, Okun, Krumin, Carandini; Steinmetz et al. *Science* 2021; CC BY-NC-SA 4.0): mouse visual cortex,
spike-sorted with pyKilosort, while the mouse viewed 112 natural images, 5 repeats each.

## 0. Setup

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # locally, uv has already installed everything
    !pip install -q bombcell

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bombcell as bc

URL = "https://ndownloader.figshare.com/files/61060780"   # AL031_2019-12-13_noRawData.tar (~1.7 GB)
DATA_DIR = Path("/content/data") if IN_COLAB else Path("../data").resolve()  # where the dataset goes
FS = 30000                                                # sampling rate (Hz)

In [ ]:
# Download and extract (~1.7 GB, a few minutes). Skipped if already there.
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not any(DATA_DIR.rglob("spike_times.npy")):
    !curl -sL {URL} | tar -x -C {DATA_DIR}

ks_dir = next(DATA_DIR.rglob("spike_times.npy")).parent
session_dir = ks_dir.parent
save_path = session_dir / "bombcell"
print("kilosort folder:", ks_dir)

## 1. Run Bombcell

Bombcell needs only the Kilosort folder. The raw data (`.ap.bin`) is optional: it improves the
amplitude and SNR metrics, but here we run on templates only.

All settings live in one dictionary, `param`.

In [ ]:
param = bc.get_default_parameters(str(ks_dir), kilosort_version=2)   # pyKilosort ~ Kilosort 2.5
param["computeDrift"] = 1                # estimate per-unit drift
param["computeDistanceMetrics"] = 0      # isolation distance / L-ratio: slower, off by default
param["splitGoodAndMua_NonSomatic"] = 0  # 1 = split non-somatic units into good / MUA

# the classification thresholds
bc.print_qm_thresholds(param) if hasattr(bc, "print_qm_thresholds") else \
    print({k: v for k, v in param.items() if k.startswith(("min", "max"))})

In [ ]:
t0 = time.time()
quality_metrics, param, unit_type, unit_type_string = bc.run_bombcell(str(ks_dir), str(save_path), param)
print(f"done in {time.time() - t0:.0f} s")

**Reading the summary plots** (more in the [wiki](https://github.com/Julie-Fabre/bombcell/wiki/Summary-output-plots)):
- **Waveform overlay:** templates of each class. Do the "noise" ones look like noise?
- **Upset plots:** which metrics exclude which units, and how they overlap.
- **Histograms:** distribution of each metric with its threshold. Is a threshold cutting through the middle of a distribution, or removing almost nothing?

## 2. Output files

In [ ]:
# Bombcell's own output folder
for f in sorted(save_path.rglob("*")):
    if f.is_file():
        print(f"{f.stat().st_size / 1e3:8.0f} kB  {f.relative_to(save_path)}")

In [ ]:
# ...and one .tsv per metric in the Kilosort folder, so Phy shows them as columns.
# cluster_bc_unitType.tsv holds the labels.
print(sorted(f.name for f in ks_dir.glob("cluster_*.tsv")))
pd.read_csv(ks_dir / "cluster_bc_unitType.tsv", sep="\t").head()

In [ ]:
# One row per unit: label + every metric
qm_table = pd.DataFrame(quality_metrics)
qm_table.insert(0, "bc_label", unit_type_string)
print(qm_table["bc_label"].value_counts())
qm_table.head()

In [ ]:
# Same table, but True/False: does each unit pass each threshold?
bc.make_qm_table(quality_metrics, param, unit_type_string).head()

In [ ]:
# Reload everything later without recomputing
param, quality_metrics, _ = bc.load_bc_results(str(save_path))

In [ ]:
# A methods paragraph for your paper, with the exact parameters you used
bc.print_methods_text(param, quality_metrics)

## 3. Look at units in the GUI

Browse units by class with the buttons. For each unit, ask: does the waveform look neuronal?
Is there a refractory period in the autocorrelogram? Is the amplitude stable over time, or cut off?

You can also label units yourself; Bombcell saves your labels so you can compare with its own.

In [ ]:
try:  # needed for widgets in Colab
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

gui = bc.unit_quality_gui(
    ks_dir=str(ks_dir),
    quality_metrics=quality_metrics,
    unit_types=unit_type,
    param=param,
    save_path=str(save_path),
)

✏️ **Exercise:** find one unit you disagree with Bombcell about. Which metric drove its label?
Use `qm_table.loc[qm_table.phy_clusterID == UNIT_ID]` to look at its numbers.

In [ ]:
UNIT_ID = 0  # change me
qm_table.loc[qm_table.phy_clusterID == UNIT_ID].T

## 4. Fine-tune the classification

Thresholds are a choice, not a truth. Relabelling is instant: no need to recompute the metrics.

In [ ]:
def relabel(**new_thresholds):
    p = dict(param, **new_thresholds)
    _, labels = bc.get_quality_unit_type(p, quality_metrics)
    return pd.Series(labels).value_counts()

pd.DataFrame({
    "default": relabel(),
    "strict RPV (5%)": relabel(maxRPVviolations=0.05),
    "lenient RPV (20%)": relabel(maxRPVviolations=0.2),
    "presence > 0.9": relabel(minPresenceRatio=0.9),
}).fillna(0).astype(int)

✏️ **Exercise:** you want to report single-neuron spike timing (e.g. cross-correlograms between pairs).
Which thresholds would you tighten? Now you want to decode the image from the population: which would you relax?

In [ ]:
# try your own
relabel(maxRPVviolations=0.1, maxPercSpikesMissing=20, minPresenceRatio=0.7, minNumSpikes=300)

## 5. Quality metrics vs. stimulus responses

Curation throws units away. Are we throwing away information? (Kenneth Harris's question.)

For each unit we measure how **reliably** it responds to the images: split the 5 repeats into two halves,
average responses to each of the 112 images in each half, and correlate the two halves.
A unit that responds consistently to the same images has a high split-half reliability.

In [ ]:
def load_any(path):
    return np.load(path).squeeze() if path.suffix == ".npy" else np.loadtxt(path).squeeze()

stim_files = {f.name.lower(): f for f in session_dir.rglob("*") if f.is_file() and f.name.lower().startswith("stim")}
print("stimulus files:", list(stim_files))
stim_times = load_any(next(f for n, f in stim_files.items() if "time" in n))
stim_ids = load_any(next(f for n, f in stim_files.items() if "id" in n)).astype(int)
print(f"{len(stim_times)} presentations of {len(np.unique(stim_ids))} images, {stim_times[0]:.1f}–{stim_times[-1]:.1f} s")

In [ ]:
spike_times = np.load(ks_dir / "spike_times.npy").squeeze() / FS
spike_units = np.load(ks_dir / "spike_templates.npy").squeeze()
unit_ids = qm_table["phy_clusterID"].to_numpy().astype(int)

# response = spike count 50-500 ms after onset minus 450 ms baseline before onset
def stim_response(st):
    resp = np.searchsorted(st, stim_times + 0.5) - np.searchsorted(st, stim_times + 0.05)
    base = np.searchsorted(st, stim_times) - np.searchsorted(st, stim_times - 0.45)
    return (resp - base) / 0.45

# repeat number of each presentation (0..4) for each image
rep = np.zeros(len(stim_ids), int)
for i in np.unique(stim_ids):
    rep[stim_ids == i] = np.arange((stim_ids == i).sum())
images = np.unique(stim_ids)

def split_half_reliability(r):
    a = np.array([r[(stim_ids == i) & (rep % 2 == 0)].mean() for i in images])
    b = np.array([r[(stim_ids == i) & (rep % 2 == 1)].mean() for i in images])
    if a.std() == 0 or b.std() == 0:
        return np.nan
    return np.corrcoef(a, b)[0, 1]

order = np.argsort(spike_units, kind="stable")
st_sorted, su_sorted = spike_times[order], spike_units[order]
bounds = np.searchsorted(su_sorted, unit_ids), np.searchsorted(su_sorted, unit_ids, side="right")
reliability = np.array([split_half_reliability(stim_response(np.sort(st_sorted[s:e]))) for s, e in zip(*bounds)])
qm_table["reliability"] = reliability
qm_table.groupby("bc_label")["reliability"].describe()

In [ ]:
# Reliability by class
colors = {"GOOD": "tab:green", "MUA": "tab:orange", "NOISE": "tab:red", "NON-SOMA": "tab:purple"}
fig, ax = plt.subplots(figsize=(5, 3))
for lab, c in colors.items():
    x = qm_table.loc[qm_table.bc_label == lab, "reliability"].dropna()
    if len(x):
        ax.hist(x, bins=np.linspace(-0.3, 1, 40), histtype="step", lw=2, color=c, label=f"{lab} (n={len(x)})")
ax.set_xlabel("split-half reliability"); ax.set_ylabel("units"); ax.legend()
plt.show()

In [ ]:
# Each quality metric vs reliability, coloured by Bombcell label
metrics = ["fractionRPVs_estimatedTauR", "percentageSpikesMissing_gaussian", "presenceRatio",
           "maxDriftEstimate", "nSpikes", "spatialDecaySlope", "waveformDuration_peakTrough", "nPeaks"]
metrics = [m for m in metrics if m in qm_table and qm_table[m].notna().any()]
fig, axs = plt.subplots(2, int(np.ceil(len(metrics) / 2)), figsize=(4 * np.ceil(len(metrics) / 2), 7))
for ax, m in zip(axs.flat, metrics):
    for lab, c in colors.items():
        d = qm_table[qm_table.bc_label == lab]
        ax.scatter(d[m], d["reliability"], s=8, c=c, alpha=0.6, label=lab)
    if m in ("nSpikes",):
        ax.set_xscale("log")
    ax.set_xlabel(m); ax.set_ylabel("reliability")
axs.flat[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

**The trade-off:** sweep one threshold and watch how many units survive, and how many *reliable* units we lose.

In [ ]:
RELIABLE = 0.3   # call a unit "stimulus-reliable" above this

def sweep(threshold_name, values):
    n_kept, n_reliable_kept = [], []
    for v in values:
        _, labels = bc.get_quality_unit_type(dict(param, **{threshold_name: v}), quality_metrics)
        keep = labels == "GOOD"
        n_kept.append(keep.sum())
        n_reliable_kept.append((keep & (reliability > RELIABLE)).sum())
    return np.array(n_kept), np.array(n_reliable_kept)

sweeps = {
    "maxRPVviolations": np.linspace(0.01, 0.5, 20),
    "maxPercSpikesMissing": np.linspace(5, 50, 20),
    "minPresenceRatio": np.linspace(0.3, 1.0, 20),
}
n_reliable_total = np.nansum(reliability > RELIABLE)
fig, axs = plt.subplots(1, len(sweeps), figsize=(4 * len(sweeps), 3.2))
for ax, (name, vals) in zip(axs, sweeps.items()):
    kept, kept_rel = sweep(name, vals)
    ax.plot(vals, kept, label="GOOD units")
    ax.plot(vals, kept_rel, label=f"GOOD & reliable > {RELIABLE}")
    ax.axhline(n_reliable_total, ls=":", c="gray", label="all reliable units")
    ax.axvline(param[name], ls="--", c="k", lw=1)
    ax.set_xlabel(name); ax.set_ylabel("units")
axs[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

**Discussion**
- Do MUA units carry stimulus information? Then for population decoding, excluding them costs you signal.
- Do noise units? If some "noise" units are reliable, look at them in the GUI: are they real neurons with odd waveforms?
- Reliability rises with firing rate, and so do several quality metrics. How would you disentangle the two?
- For single-neuron claims (tuning of *one* cell, spike timing), contamination matters much more than for population analyses.

**Take-home:** pick thresholds for your question, report them (`print_methods_text`), and check what your curation removes.